In [ ]:
# Package Imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binom
from math import comb


rng = np.random.default_rng() # Random Number Generator

# The Dominant Distribution

In this lesson, we will explore why a system with many equally likely microscopic arrangements can nevertheless have a strongly preferred **macroscopic** outcome. This idea will prepare us to think about the Boltzmann distribution.

By the end, you should be able to:

- distinguish a microstate from a macrostate
- explain why some macrostates are more probable than others
- explain why the relative spread of a large system becomes very small and has a dominant distribution
- count the microstates associated with an energy-distribution macrostate
- simulate energy exchange in a system with fixed total energy

## A physical coin-flip experiment

Each student should flip a coin **10 times** and record the number of heads. A set of 10 flips is one *trial*.

Enter the class results in the cell below. Each number should be the number of heads from one student's 10 flips.

In [ ]:
# List the number of heads each student got in 10 flips
class_heads = [] 

In [ ]:
n_coins = 10
possible_heads = np.arange(n_coins + 1)

plt.figure(figsize=(7, 4))
plt.hist(class_heads, bins=np.arange(-0.5, n_coins + 1.5, 1), edgecolor='black', align='mid')
plt.xticks(possible_heads)
plt.xlabel('Number of heads in 10 flips')
plt.ylabel('Number of students')
plt.title(f'Class coin-flip results: 10 flips per student (n = {len(class_heads)} students)')
plt.show()

**Discuss:** Which result appears most common? Are all possible numbers of heads equally likely? Why might a result near five heads occur more often than zero or ten heads?

## Start small: list every outcome for three coins

A particular ordered result such as `HTH` is a **microstate**. The statement "two heads" is a **macrostate**: it does not specify which flips were heads.

With three coins, we can list every possible microstate explicitly.  In your lab notebook or on paper:
* Make a list of every possible microstate (e.g. `HTH`). 
* For each possible microstate list the number of heads (the macrostate classification) in that microstate.
* Discuss:
    * Is each microstate equally likely?
    * Is each macrostate equally likely?
* Calculate the probability of observing each macrostate.

## Multiplicity grows quickly

For N coins, the number of microstates with exactly k heads is written

$\binom{N}{k}$

For now, read this as: **the number of sequences containing exactly (k) heads in (N) flips.** The cell below calculates these counts.

`comb` is a function from the math module that calculates the number of combinations (binomial coefficient) of N items taken k at a time.

You can calculate the total number of possible microstates for this system as 2$^N$.

In [ ]:
N = 3
k = 2

number_of_microstates = comb(N, k) 
total_microstates = 2**N

probability_of_macrostate = number_of_microstates / total_microstates

print(f'Number of microstates for {k} heads in {N} flips: {number_of_microstates}')
print(f'Total number of microstates for {N} flips: {total_microstates}')
print(f'Probability of macrostate: {probability_of_macrostate}')

Using the code above as a guide, write code to calculate the probability of getting each number of heads (0, 1, 2, ...) for 10 coin flips.  You need to do the calculation for each number of heads, so you will want to use a `for` loop.  

Make a bar plot using `plt.bar(x_values, y_values)` to show the probability as a function of the number of heads in a trial of 10 coin flips.

After you make your plot, compare it to the histogram of the class generated data. What is the relationship between these two graphs? How well do the two plots agree with each other? 

In [ ]:
# your code goes here
N = 10
ks = np.asarray([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

## Simulate many trials

We now simulate many independent trials, each consisting of 10 fair coin flips. In the statistical library, `binom.rvs` generates random outcomes from a binomial distribution. This allows you to simulate flipping a coin many times with a single line of code.

The cell below simulates 20 people each flipping 10 coins and plots a histogram showing how many times each number of heads appears. Do the following:
* Before you modify anything, run the cell several times and observe the distributions. For 10 trials, how predictable is the distribution?
* Try increasing the number of trials from 10. How many trials do you need to get a predictable distribution?

In [ ]:
n_coins = 10        # Number of coins to flip
p_heads = 0.5       # Probability of getting heads on a single flip
n_trials = 10       # Number of trials to simulate

simulated_heads = binom.rvs(n=n_coins, p=p_heads, size=n_trials, random_state=rng)
fraction_heads = simulated_heads / n_coins

plt.hist(fraction_heads, bins=50, edgecolor='black', align='mid')
plt.xlim(0, 1)
plt.xlabel('Fraction of flips that are heads')
plt.ylabel('Number of trials')
plt.title(f'{n_trials} trials of {n_coins} coin flips')
plt.show()


Once you have reached a high enough number of trials, you should see that the distribution is predictable. Even still, with just 10 coins over many trials, we will still see a large fraction of the coin flips that are not 50% heads. 

We can use `binom.cdf` (cumulative distribution function) to compute how many trials we expect to result in close to 50% heads. The `cdf(k,n,p)` computes the probability of getting up to `k` successes for `n` trials with a probability of `p` on each trial (50% for our fair coin). By using the difference between two cdf functions, we can determine the probability within a range.

In [ ]:
n_coins = 10 

lower = int(np.ceil(0.49 * n_coins)) # Calculate the lower bound for 49% of n_coins
upper = int(np.floor(0.51 * n_coins)) # Calculate the upper bound for 51% of n_coins

# Calculate the probability of getting between 49% and 51% heads
probability_near_half = binom.cdf(upper, n=n_coins, p=0.5) - binom.cdf(lower - 1, n=n_coins, p=0.5) 

print(f'{n_coins:4d} coins: P(49% to 51% heads) = {probability_near_half:.3f}')

If you flip 10 coins, you can expect that about 25% of the time the number of coin flips that will be heads will fall between 49 and 51%.

## A large system: the dominant region

Now we will explore what happens as we increase the number of coins. Write code that plots histograms for each of the coin counts listed below; you can do this most efficiently with a for loop. Do the following (note these are the same instructions used to build the example above):
* Simulate 100,000 trials of flipping the specified number of coins using the binom.rvs function. Use a fair coin (p=0.5).
* The histograms should show the number trials in which you got each fraction of coin flips that were heads.
* To be able to compare the histograms visually, use the same number of bins for each (e.g. 50) and the same x-axis limits (0 to 1).

In [ ]:
coin_counts = [100, 1000, 10_000, 100_000]
n_trials = 100_000

# Your code goes here


After creating the histograms use the cumulative distribution function `binom.cdf` to calculate for each number of coins in `coin_counts` what the probability is that the number of coin flips will fall between 49 and 51%.

In [ ]:
# Your code goes here

Discussion Questions:
* As the number of coins flipped increases, what happens to the number of available microstates?
* With a very large number of coins flipped, is each microstate equally likely?
* As the number of coins increases, what happens to the likelihood that the observed macrostate will be close to 50% heads?
* For a large number of coins flipped, what is the probabiliy that the fraction of heads will be significantly different than 0.5?
* It is theoretically possible to flip a fair coin and get heads every time. Why is this effectively impossible for large numbers of coins flipped?
* Explain why a large system can have a predictable macroscopic behavior even though the individual microscopic components behave randomly.

## Connection to Statistical Mechanics: Energy Exchange

Next, we will explore what the distribution of energy among particles looks like when we allow the particles to freely and randomly exchange energy. We will start with an in-person game to model the random exchange of energy among particles while keeping the total energy fixed. Then we will model much larger systems to examine what the dominant distribution of energies looks like. 

## Energy Exchange Game

This activity adapts Hanson and Michalek's [“Give Them Money: The Boltzmann Game” (*Journal of Chemical Education*, 2006, 83, 581)](https://doi.org/10.1021/ed083p581). We use tokens as identical energy quanta: each student represents a distinguishable particle, and the whole class is the isolated fixed-energy system (it will not exchange energy with its surroundings).

### Predict before playing

Each participant will start with one token. Before any exchanges, sketch your prediction for the eventual histogram: horizontal axis = quanta per person; vertical axis = number of people.

- Will everyone tend to return to one quantum, or will the energies redistribute in some way?
- How many people do you predict will have zero, one, two, or more quanta?
- What must stay constant? Could one person ever hold every quantum?

Record your prediction before collecting results.

### Play and record

1. Form facing pairs in two circles. With an odd class size, your instructor will play.
2. Each pair plays rock–paper–scissors once per round.
3. The loser transfers one token to the winner if the loser has any. A tie or an empty-handed loser causes no transfer. Do not replay ties.
4. Students with zero tokens keep playing and can gain tokens.
5. Rotate one circle to change partners each round.
6. Pause every few rounds. Tally everyone by exact token count, including zero, and record the round number.

Use the same participants and token supply throughout this run. The game treats both directions of exchange as equally likely; it assumes students choose rock, paper, and scissors without systematic winning advantages.

In [ ]:
# Your instructor will guide you through tallying the number of students with 0, 1, 2, ... quanta at each timepoint.

timepoints = ["Initial", "First Checkpoint", "Second Checkpoint"]
# Each list gives the number of students with 0, 1, 2, ... quanta.
counts = [
    [], # Put a list of counts for the first timepoint here in the format [count_0, count_1, count_2, ...]
    [], # Put a list of counts for the second timepoint here
    [] # Put a list of counts for the third timepoint here in the
    ]


In [ ]:
# Run this after entering values in the previous cell.

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for ax, timepoint, tally in zip(axes, timepoints, counts):
    ax.bar(range(len(tally)), tally)
    ax.set_title(timepoint)
    ax.set_xlabel("Quanta per student")
    ax.set_xticks(range(len(tally)))

axes[0].set_ylabel("Number of students")
plt.tight_layout()
plt.show()


**Compare with your prediction:** What changed while the mean stayed fixed? Did later histograms resemble each other even though individuals kept gaining and losing tokens? Record another late-round tally to see the fluctuations. Equilibrium describes the statistics of the distribution; exchanges continue at equilibrium.

## Counting microstates and macrostates

To think about why the distribution appears the way that it does, we will consider the number of microstates for each macrostate. We will do it by hand for a very small system: three quanta among three particles.

We will consider the particles as distinguishable, but the quanta as indistinguishable. A **microstate** specifies each particle's energy, in order: `(2, 1, 0)` and `(1, 2, 0)` are different microstates. We do not label the quanta; swapping two tokens without changing any particle's energy creates no new microstate.

A **macrostate** specifies the distribution without assigning the energies to named particles. Here we write its energies in descending order, so both microstates above belong to the macrostate `(2, 1, 0)`. This macrostate has one particle at zero, one at one, and one at two quanta.

| Macrostate: energies, ignoring names | Number of microstates (your count) | 
| --- | --- | 
| `(3, 0, 0)` | | 
| `(2, 1, 0)` | | 
| `(1, 1, 1)` | | 

1. List every distinct ordered assignment of quanta to each of three particles for each row. Why is `(1, 1, 1)` counted just once?
2. Add your counts. Which macrostate has the most microstates?
3. If all allowed microstates are equally probable at equilibrium, what is the probability of each macrostate?
4. Is equal sharing the most probable macrostate? Does a fair exchange rule require equal energies?

## Modeling a large number of exchanges

Write code that simulates the random exchange of energy among a large number of particles with many quanta. 

* Create an array where each position in the array represents one molecule. The value in the array describes the number of quanta that molecule has. A good starting place is several hundred molecules. Remember that you can use `np.zeroes(N)` to create an array of length N filled with zeroes.
* Choose an initial number of quanta to give each molecule. A good starting place is a few quanta per molecule. You can fill an existing array called arr with the number stored in `value` using `arr.fill(value)`.
* Use a for range loop to simulate many potential interactions. In each iteration of the simulation:
    * Randomly select a molecule to donate energy. You can randomly select an index in your molecule array.
    * Randomly select a molecule to accept energy.
    * Remove a quantum of energy from the donor and give it to the acceptor. Make sure you check that your donor has more than 0 quanta before donating!
* After many steps, plot a histrogram showing what the distribution of energy looks like. The x-axis should be the number of quanta and the y-axis should be the count of molecules with that energy.
* Update your simulation to ensure that you have enough simulation steps to arrive at a consistent, repeatable distribution. If desired you can tweak your number of molecules and quanta as well.
* Repeat your simulation, but this time for your initial condition put all of the quanta on a single molecule. Make sure you use the same number of molecules and the same total number of quanta.

For the two simulations you ran, you started with very different non-equilibrium conditions (energy evenly distributed and energy all on one molecule), but you should have arrived at the same distribution of energy (i.e. the same macrostate) from random energy exchanges. This most probably distribution is the Boltzmann distribution for evenly spaced energy levels. Note that because we're using relatively small numbers of molecules and quanta to allow equilibration without a simulation that runs for a very long time, the two histograms may not be identical, but they should be qualitatively the same distribution.

For your report, you will submit one of your histograms.

In [ ]:
# This code block demonstrates how to randomly select two distinct integers from 0 to 9 using NumPy's random number generator.
# The `choice` function is used with `replace=False` to ensure that the two integers are unique.

int1, int2 = rng.choice(10, size=2, replace=False)
print(f'Random integers: {int1}, {int2}')

In [ ]:
# Your code goes here

# Number of molecules in the simulation
N = 500
# Number of quanta per molecule
q = 4